# 03 — Results Analysis & Policy Implications
Deep-dive into optimization results:
1. Scenario comparison and key findings
2. Sensitivity analysis (solar × wind cost grid)
3. Technology learning curve impact
4. Policy implications for NYISO 2030 planning
5. Counterintuitive result: why 100% ZCE is cheaper than 80% RPS


In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from models.optimizer import NYEnergyOptimizer
from data.generate_data import generate_price_projections
from config import TECH_COLORS, SCENARIO_COLORS, SCENARIOS, NY_BASELINE_CO2_MT

opt = NYEnergyOptimizer()
results = opt.run_all_scenarios()
scenario_map = {r.scenario_name: r for r in results}
projections = generate_price_projections()

## 1. Scenario Comparison — Key Metrics

In [ ]:
summary = pd.DataFrame([{
    "Scenario": r.scenario_name,
    "Cost ($B/yr)": r.total_cost_b_yr,
    "LCOE ($/MWh)": r.lcoe_system,
    "Renewable %": f"{r.renewable_fraction:.0%}",
    "Zero-Carbon %": f"{r.zero_carbon_fraction:.0%}",
    "CO₂ (Mt/yr)": r.co2_mt_yr,
    "CO₂ Reduction": f"{r.co2_reduction_pct:.0%}",
} for r in results])
print(summary.to_string(index=False))

## 2. Sensitivity Analysis — Solar × Wind Cost Grid

In [ ]:
solar_range = list(np.linspace(500, 1150, 8, dtype=int))
wind_range  = list(np.linspace(800, 1500, 8, dtype=int))

print(f"Running {len(solar_range) * len(wind_range)} sensitivity scenarios for '80% Renewable'...")
sens_df = opt.run_sensitivity("80% Renewable", solar_range, wind_range)
print(f"Cost range: ${sens_df['total_cost_b_yr'].min():.2f}B – ${sens_df['total_cost_b_yr'].max():.2f}B/yr")
print(f"LCOE range: ${sens_df['lcoe_system'].min():.0f} – ${sens_df['lcoe_system'].max():.0f}/MWh")
sens_df.head(10)

In [ ]:
pivot = sens_df.pivot_table(
    values="total_cost_b_yr", index="solar_capital_per_kw", columns="wind_capital_per_kw"
)
fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[f"${c}" for c in pivot.columns],
    y=[f"${r}" for r in pivot.index],
    colorscale="RdYlGn_r",
    text=[[f"${v:.1f}B" for v in row] for row in pivot.values],
    texttemplate="%{text}",
    colorbar=dict(title="$B/yr"),
))
fig.update_layout(
    title="Sensitivity: Total System Cost ($B/yr) — 80% RPS Scenario",
    xaxis_title="Wind Capital Cost ($/kW)",
    yaxis_title="Solar Capital Cost ($/kW)",
    height=400,
)
fig.show()

## 3. Why 100% ZCE Is Cheaper Than 80% RPS

In [ ]:
print("=== Technology cost drivers ===")
print()
r50 = scenario_map["50% Renewable"]
r80 = scenario_map["80% Renewable"]
r100 = scenario_map["100% Zero-Carbon"]

offshore_cost_80 = r80.cost_breakdown_m.get("Offshore Wind", 0) / 1e3
offshore_gw_80 = r80.capacity_gw.get("Offshore Wind", 0)
print(f"80% RPS offshore wind: {offshore_gw_80:.1f} GW costing ${offshore_cost_80:.2f}B/yr")
print(f"  This is {offshore_cost_80/r80.total_cost_b_yr:.0%} of total 80% RPS cost")
print()
print(f"100% ZCE: offshore wind = {r100.capacity_gw.get('Offshore Wind',0):.0f} GW")
print(f"          nuclear       = {r100.capacity_gw.get('Nuclear',0):.1f} GW @ $91/MWh LCOE")
print(f"          solar         = {r100.capacity_gw.get('Solar PV (Utility)',0):.1f} GW")
print()
print(f"Cost comparison: 80% RPS ${r80.total_cost_b_yr:.2f}B vs 100% ZCE ${r100.total_cost_b_yr:.2f}B")
print(f"  100% ZCE is {(r100.total_cost_b_yr/r80.total_cost_b_yr - 1):.0%} the cost of 80% RPS")

## 4. CLCPA Target Alignment

In [ ]:
clcpa_targets = {
    "Offshore Wind": 9.0,   # GW by 2035
    "Solar PV (Utility)": 18.0,  # GW by 2030
    "Battery Storage (4h)": 6.0,  # GW by 2030
}
print("CLCPA capacity targets vs. modeled optimal:")
print()
print(f"{'Technology':<25} {'CLCPA Target':>15} {'50% RPS':>10} {'80% RPS':>10} {'100% ZCE':>10}")
print("-" * 75)
for tech, target in clcpa_targets.items():
    vals = [scenario_map[s].capacity_gw.get(tech, 0) for s in SCENARIOS]
    flags = ["✓" if v >= target else "✗" for v in vals]
    print(f"  {tech:<23} {target:>12.0f} GW "
          f"  {vals[0]:>5.1f}{flags[0]} "
          f"  {vals[1]:>5.1f}{flags[1]} "
          f"  {vals[2]:>5.1f}{flags[2]}")

## 5. CO₂ Abatement Cost Analysis

In [ ]:
baseline_co2 = NY_BASELINE_CO2_MT
r_ref = results[0]

print("Marginal abatement cost (vs 50% RPS baseline):")
print()
for r in results[1:]:
    delta_cost_m = (r.total_cost_b_yr - r_ref.total_cost_b_yr) * 1e9  # $/yr
    delta_co2 = (r_ref.co2_mt_yr - r.co2_mt_yr) * 1e6  # tonnes/yr
    abatement = delta_cost_m / delta_co2 if delta_co2 > 0 else float("inf")
    print(f"  {r.scenario_name}: ${abatement:.0f}/tonne CO₂ abated")
    print(f"    Δ Cost: +${(r.total_cost_b_yr - r_ref.total_cost_b_yr):.2f}B/yr")
    print(f"    Δ CO₂: -{(r_ref.co2_mt_yr - r.co2_mt_yr):.1f} Mt/yr")
    print()